# TrafficFlowBench: When Does Temporal Repair Fail?

TrafficFlowBench Task 1 hides selected five-minute speed and flow readings. This study compares simple repairs on the same hidden cells across five synthetic freeway directions and a separate January window.

**TL;DR**

- Temporal scores **0.88–0.92** across five corridors, versus **0.53–0.83** for the 28-day historical mean and **0.75–0.84** for Spatial.
- Its score falls from short to long gaps in every corridor; the gap table shows each short minus long difference.
- Day-edge cells are a weak spot: without readings on both sides within that day, Temporal uses the earlier profile. Each day is repaired from its own masked file.
- Switching to Spatial on long gaps does not help in four corridors. D7_I405_N reverses the raw order (0.8530 versus 0.8551), but excluding day-edge cells flips it back (0.8591 versus 0.8544).

**Scope:** D12_I405_N has 8 February held-out days; D12_I5_N, D7_I405_N, D12_I5_S, and D7_I10_E have 20 each. That is 1,359,715 paired February target cells. A separate D12_I405_N January backtest has 12 days and 74,431 cells. The records are **synthetic**. These are **subset diagnostics, not competition scores**; the published scorer matched recorded Temporal predictions on all 1,434,146 selected cells.

**Baseline caveat:** The seven-day historical mean is likely weaker than the organizers' full-data baseline and can overstate Temporal's advantage. The 28-day version is a stronger **partial-history** comparison, not the full-data baseline.

**Metric:** The diagnostic combines normalized speed and per-lane flow errors, then averages R1/R2/R3; **higher is better and 1.0 is a perfect repair**.

> **Terms in plain language**
>
> **Sensor gap:** a missing five-minute speed and flow reading. **Mask:** the published layer with selected readings hidden. **Corridor:** one freeway direction. **Historical mean:** an earlier-days average for a link at the same weekday and time.
>
> **Temporal:** fill from observed readings before and after the gap on the same link. **Spatial:** fill from observed neighboring links at the same time. **Fallback:** use Temporal for runs up to six steps, then Spatial for longer runs, separately by channel.
>
> **R1/R2/R3:** published masks hiding 20%/30%/50% of eligible cells. **Paired difference:** compare repairs on the same hidden cells. **Chronological validation:** learn from earlier days and score later days.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
opening_scores = {'D12_I405_N': {'Historical mean (7 days)': 0.642583, 'Historical mean (28 days)': 0.833895, 'Spatial': 0.775023, 'Temporal/spatial fallback': 0.912013, 'Temporal': 0.922221}, 'D12_I5_N': {'Historical mean (7 days)': 0.398458, 'Historical mean (28 days)': 0.554345, 'Spatial': 0.843477, 'Temporal/spatial fallback': 0.905539, 'Temporal': 0.908553}, 'D7_I405_N': {'Historical mean (7 days)': 0.28413, 'Historical mean (28 days)': 0.526557, 'Spatial': 0.840352, 'Temporal/spatial fallback': 0.881731, 'Temporal': 0.881763}, 'D12_I5_S': {'Historical mean (7 days)': 0.495917, 'Historical mean (28 days)': 0.633272, 'Spatial': 0.747911, 'Temporal/spatial fallback': 0.891983, 'Temporal': 0.905039}, 'D7_I10_E': {'Historical mean (7 days)': 0.550512, 'Historical mean (28 days)': 0.67153, 'Spatial': 0.826762, 'Temporal/spatial fallback': 0.903888, 'Temporal': 0.911162}}
panels = ['D12_I405_N', 'D12_I5_N', 'D7_I405_N', 'D12_I5_S', 'D7_I10_E']
methods = ['Historical mean (7 days)', 'Historical mean (28 days)', 'Spatial', 'Temporal']
colors = ['#b6c1c5', '#839da5', '#cf8b58', '#137c77']
fig, ax = plt.subplots(figsize=(10, 5.4))
y = np.arange(len(panels)); h = .19
for j, method in enumerate(methods):
    ax.barh(y + (j-1.5)*h, [opening_scores[p][method] for p in panels],
            height=h, label=method, color=colors[j])
ax.set_yticks(y, panels)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel('Equal-regime Task 1 subset diagnostic (higher is better)')
ax.set_title('Temporal repair beats even a 28-day history profile on every corridor')
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## Setup

Use release 1.0 rebuilt on 2026-09-11, contract 2.0.0, the published masks, and fixed seeds. On Kaggle, attach the competition data. Locally, the selected per-file mirror is used. Every scored day follows its profile days.

In [2]:
from pathlib import Path
import json, platform
import numpy as np
import pandas as pd
from IPython.display import display

SEED = 7
np.random.seed(SEED)
# Set VERIFY = False when changing corridors or date windows; data-integrity checks still run.
VERIFY = True
BASE_PANEL = 'D12_I405_N'
PANELS = ['D12_I405_N', 'D12_I5_N', 'D7_I405_N', 'D12_I5_S', 'D7_I10_E']
KEY = ['timestamp', 'station_id', 'link_id']
CHANNELS = ['speed_kmh', 'flow_vph']
PROFILE_7 = pd.date_range('2031-02-02', '2031-02-08').strftime('%Y-%m-%d').tolist()
PROFILE_28 = pd.date_range('2031-01-12', '2031-02-08').strftime('%Y-%m-%d').tolist()
FEB_8 = pd.date_range('2031-02-09', '2031-02-16').strftime('%Y-%m-%d').tolist()
FEB_20 = pd.date_range('2031-02-09', '2031-02-28').strftime('%Y-%m-%d').tolist()
FEB_DAYS = {p: FEB_8 if p == BASE_PANEL else FEB_20 for p in PANELS}
JAN_7 = pd.date_range('2031-01-13', '2031-01-19').strftime('%Y-%m-%d').tolist()
JAN_28 = pd.date_range('2030-12-23', '2031-01-19').strftime('%Y-%m-%d').tolist()
JAN_HELD = pd.date_range('2031-01-20', '2031-01-31').strftime('%Y-%m-%d').tolist()

available = sorted(['D7_I10_E','D7_I10_W','D7_I210_E','D7_I210_W',
                    'D7_I405_N','D7_I405_S','D12_I5_N','D12_I5_S','D12_I405_S'])
if VERIFY:
    assert np.random.default_rng(20261002).choice(available, size=2, replace=False).tolist() == ['D12_I5_N','D7_I405_N']
remaining = sorted(set(available) - {'D12_I5_N','D7_I405_N'})
if VERIFY:
    assert np.random.default_rng(20261003).choice(remaining, size=len(remaining), replace=False).tolist()[:3] == ['D12_I5_S','D7_I210_E','D7_I10_E']

LOCAL_BASE = Path('D:/kaggle-sprint/data/traffic')
PROJECT_DATA = Path('D:/kaggle-sprint/trafficflow_sensor_gaps/data')
LOCAL_MODE = LOCAL_BASE.exists()
if LOCAL_MODE:
    contract_path = LOCAL_BASE / 'competition_contract.json'
else:
    roots = [p for p in Path('/kaggle/input').glob('**/kaggle_public')
             if (p / 'config' / 'competition_contract.json').exists()]
    assert len(roots) == 1, f'Expected one attached release; found {len(roots)}'
    release = roots[0]
    contract_path = release / 'config' / 'competition_contract.json'
contract = json.loads(contract_path.read_text())
assert contract['contract_version'] == '2.0.0'
assert contract['tasks']['state']['masking_regimes'] == {'R1': .2, 'R2': .3, 'R3': .5}
assert contract['data_quality']['score_threshold_pct_observed'] == 75
print('Release 1.0 rebuilt 2026-09-11 | contract', contract['contract_version'])
print('Python', platform.python_version(), '| pandas', pd.__version__,
      '| numpy', np.__version__, '| matplotlib', plt.matplotlib.__version__)

def day_path(panel, day, masked, regime=None):
    name = 'synthetic_mainline_' + day.replace('-', '_') + '.parquet'
    if masked:
        assert regime in {'R1', 'R2', 'R3'}
        relative = Path('corridors') / panel / 'train' / 'mainline_states_masked' / ('mask_regime=' + regime) / name
    else:
        relative = Path('corridors') / panel / 'train' / 'mainline_states' / ('year_month=' + day[:7]) / name
    if not LOCAL_MODE:
        return release / relative
    candidate = PROJECT_DATA / relative
    if candidate.exists():
        return candidate
    assert panel == BASE_PANEL
    return LOCAL_BASE / Path(*relative.parts[1:])

def template_path(panel):
    if not LOCAL_MODE:
        return release / 'task1' / panel / 'train' / 'sample_submission_state.csv'
    if panel == BASE_PANEL:
        return LOCAL_BASE / 'sample_submission_state.csv'
    return PROJECT_DATA / 'task1' / panel / 'train' / 'sample_submission_state.csv'

def lanes_path(panel):
    if not LOCAL_MODE:
        return release / 'corridors' / panel / 'network' / 'fd_parameters.csv'
    if panel == BASE_PANEL:
        return LOCAL_BASE / 'fd_parameters.csv'
    return PROJECT_DATA / 'corridors' / panel / 'network' / 'fd_parameters.csv'

def read_day(panel, day, masked, regime=None):
    path = day_path(panel, day, masked, regime)
    assert path.exists(), f'Missing selected release file: {path}'
    return pd.read_parquet(path)

### Targets

The official Task 1 template defines which blanked cells are scored; an arbitrary null in the masked layer is not a target. R1, R2, and R3 hide 20%, 30%, and 50% of eligible cells. Scoring requires at least 75% observed within the source interval and present speed and flow truth. Template keys must be unique and must point to masked blanks.

In [3]:
def load_targets(panel, held_days):
    parts = []
    for chunk in pd.read_csv(template_path(panel), usecols=KEY + ['mask_regime'], chunksize=200_000):
        part = chunk[chunk.timestamp.str[:10].isin(held_days)]
        if len(part):
            parts.append(part)
    target = pd.concat(parts, ignore_index=True)
    assert not target.duplicated(KEY).any()
    assert set(target.timestamp.str[:10]) == set(held_days)
    target['day'] = target.timestamp.str[:10]
    assert target.groupby('day').mask_regime.nunique().eq(1).all()
    regime_map = target.groupby('day').mask_regime.first().to_dict()
    assert set(regime_map.values()) == {'R1', 'R2', 'R3'}
    by_day = {day: target[target.day.eq(day)][KEY + ['mask_regime']].reset_index(drop=True)
              for day in held_days}
    return by_day, regime_map

## Repairs

The seven-day historical mean is a weekday-and-five-minute profile per link, with a link-wide earlier-days fallback. Temporal interpolates along the same link; Spatial interpolates across links in released milepost order at the same time. The fixed fallback uses Temporal for missing runs up to six five-minute steps and Spatial for longer runs, separately for each channel. A repair without bracketing observations uses the earlier profile at that edge. Task 1 is offline, so both sides of a same-day gap are available. Repairs receive only that day's masked file and earlier eligible profile observations; neighboring held-out days are not used.

In [4]:
def profile_for(panel, profile_days):
    pieces = []
    for day in profile_days:
        frame = pd.read_parquet(day_path(panel, day, False),
                                columns=['timestamp', 'link_id', 'pct_observed'] + CHANNELS)
        pieces.append(frame[(frame.pct_observed >= 75) & frame.speed_kmh.notna() & frame.flow_vph.notna()])
    history = pd.concat(pieces, ignore_index=True)
    ts = pd.to_datetime(history.timestamp, utc=True)
    history['weekday'] = ts.dt.weekday
    history['slot'] = ts.dt.hour * 12 + ts.dt.minute // 5
    profile = history.groupby(['link_id','weekday','slot'])[CHANNELS].mean()
    link_mean = history.groupby('link_id')[CHANNELS].mean()
    return profile, link_mean

def historical_for(masked, profile, link_mean):
    x = masked[['timestamp','link_id']].copy()
    ts = pd.to_datetime(x.timestamp, utc=True)
    x['weekday'], x['slot'] = ts.dt.weekday, ts.dt.hour*12+ts.dt.minute//5
    out = x.join(profile, on=['link_id','weekday','slot'])
    for ch in CHANNELS:
        out[ch] = out[ch].fillna(out.link_id.map(link_mean[ch]))
    return out[CHANNELS].set_axis(masked.index)

def gap_lengths(frame):
    result = pd.DataFrame(index=frame.index, columns=frame.columns, dtype=float)
    for link in frame.columns:
        missing = frame[link].isna()
        result[link] = missing.groupby((~missing).cumsum()).transform('sum').to_numpy()
    return result

def repair_masked(masked, ordered_links, profile, link_mean):
    times = sorted(masked.timestamp.unique())
    out = {name: {} for name in ['Historical mean','Temporal','Spatial','Temporal/spatial fallback']}
    earlier = historical_for(masked, profile, link_mean)
    for ch in CHANNELS:
        a = masked.pivot(index='timestamp', columns='link_id', values=ch).reindex(index=times, columns=ordered_links)
        h = masked.assign(prior=earlier[ch]).pivot(index='timestamp', columns='link_id', values='prior').reindex(index=times, columns=ordered_links)
        temporal = a.interpolate(axis=0, limit_area='inside').fillna(h)
        spatial = a.T.interpolate(axis=0, limit_area='inside').T.fillna(h)
        short = gap_lengths(a) <= 6
        fallback = temporal.where(short, spatial).fillna(h)
        for name, matrix in [('Historical mean',h),('Temporal',temporal),('Spatial',spatial),('Temporal/spatial fallback',fallback)]:
            lookup = matrix.stack().rename(ch)
            ix = pd.MultiIndex.from_frame(masked[['timestamp','link_id']])
            out[name][ch] = lookup.reindex(ix).to_numpy()
    return {name: pd.DataFrame(channels, index=masked.index) for name,channels in out.items()}

def published_gap_steps(masked):
    matrix = masked.pivot(index='timestamp', columns='link_id', values='speed_kmh')
    return gap_lengths(matrix).stack().rename('gap_steps')

def temporal_profile_edges(masked):
    edges = []
    for ch in CHANNELS:
        matrix = masked.pivot(index='timestamp', columns='link_id', values=ch)
        seen_before = matrix.notna().cummax()
        seen_after = matrix.notna().iloc[::-1].cummax().iloc[::-1]
        edges.append((~(seen_before & seen_after)).stack().rename(ch+'_edge'))
    out = pd.concat(edges, axis=1)
    out['edge_any'] = out[[ch+'_edge' for ch in CHANNELS]].any(axis=1)
    flow_matrix = masked.pivot(index='timestamp', columns='link_id', values='flow_vph')
    out['flow_gap_steps'] = gap_lengths(flow_matrix).stack()
    return out

## Scoring

The Task 1 component uses speed RMSE divided by 25 and per-lane flow RMSE divided by 600. Each is clipped to a score from 0 to 1; speed has weight 0.54 and flow 0.46. R1, R2, and R3 scores are averaged equally. Every method uses the same target keys, lane counts, and truth. Unmasked held-out readings are opened only after all five predictions for that day are built.

In [5]:
def component_score(frame):
    if frame.empty:
        return np.nan
    speed_rmse = np.sqrt(frame.se_speed.mean())
    flow_rmse = np.sqrt(frame.se_flow_lane.mean())
    return .54*max(0, 1-speed_rmse/25) + .46*max(0, 1-flow_rmse/600)

def evaluate_panel(panel, profile7_days, profile28_days, held_days):
    assert max(profile7_days) < min(held_days)
    assert max(profile28_days) < min(held_days)
    targets_by_day, regime_map = load_targets(panel, held_days)
    profile7, mean7 = profile_for(panel, profile7_days)
    profile28, mean28 = profile_for(panel, profile28_days)
    lane_map = pd.read_csv(lanes_path(panel)).set_index('link_id').lanes
    rows = []
    for day in held_days:
        masked = read_day(panel, day, True, regime_map[day])
        day_targets = targets_by_day[day][KEY]
        assert not masked.duplicated(KEY).any()
        marked = masked.merge(day_targets.assign(target=True), on=KEY, how='left', validate='one_to_one')
        is_target = marked.target.eq(True).to_numpy()
        assert is_target.sum() == len(day_targets)
        assert marked.loc[is_target, CHANNELS].isna().all().all()
        order = masked[['link_id','milepost']].drop_duplicates().sort_values('milepost').link_id.tolist()
        assert len(order) == masked.link_id.nunique()
        repairs = repair_masked(masked, order, profile7, mean7)
        repairs['Historical mean (28 days)'] = historical_for(masked, profile28, mean28)
        # Scoring truth is opened only after all repairs are computed from masked inputs.
        truth = read_day(panel, day, False)[KEY + CHANNELS + ['is_score_eligible']]
        joined = marked.loc[is_target, KEY].merge(truth, on=KEY, validate='one_to_one')
        assert len(joined) == len(day_targets) and joined.is_score_eligible.eq(True).all()
        assert joined[CHANNELS].notna().all().all()
        selected = pd.MultiIndex.from_frame(masked[KEY]).isin(pd.MultiIndex.from_frame(joined[KEY]))
        assert selected.sum() == len(joined)
        pred_keys = masked.loc[selected, KEY].reset_index(drop=True)
        gap_ix = pd.MultiIndex.from_frame(pred_keys[['timestamp','link_id']])
        pred_keys['gap_steps'] = published_gap_steps(masked).reindex(gap_ix).to_numpy()
        edge_here = temporal_profile_edges(masked).reindex(gap_ix)
        assert edge_here.notna().all().all()
        for col in edge_here:
            pred_keys[col] = edge_here[col].to_numpy()
        assert pred_keys.gap_steps.notna().all() and (pred_keys.gap_steps >= 1).all()
        for method, pred in repairs.items():
            frame = pd.concat([pred_keys, pred.loc[selected, CHANNELS].reset_index(drop=True)], axis=1)
            frame = frame.merge(joined, on=KEY, suffixes=('_pred','_true'), validate='one_to_one')
            assert np.isfinite(frame[['speed_kmh_pred','flow_vph_pred']].to_numpy()).all()
            frame['day'], frame['regime'], frame['method'], frame['panel'] = day, regime_map[day], method, panel
            lanes_here = frame.link_id.map(lane_map)
            assert lanes_here.notna().all()
            frame['se_speed'] = (frame.speed_kmh_pred-frame.speed_kmh_true)**2
            frame['se_flow_lane'] = ((frame.flow_vph_pred-frame.flow_vph_true)/lanes_here)**2
            rows.append(frame[['panel','day','regime','method','timestamp','link_id',
                               'gap_steps','flow_gap_steps','speed_kmh_edge','flow_vph_edge','edge_any',
                               'se_speed','se_flow_lane']])
    return pd.concat(rows, ignore_index=True), regime_map, targets_by_day

## Five-corridor results

D12_I405_N was chosen for the first test. Before seeing outcomes, seed 20261002 sampled D12_I5_N and D7_I405_N from the other nine directions. Seed 20261003 next chose D12_I5_S and D7_I210_E. The published template showed zero R2 days for D7_I210_E in the fixed February window; the next seeded direction, D7_I10_E, replaced it **before outcome scores were read**. No corridor was selected by score.

D12_I405_N has eight held-out days. The other four have twenty because the initially planned twelve-day window gave D12_I5_N only one R2 day; the longer window was fixed before outcomes. Each February corridor uses the same earlier seven- and 28-day profiles, with the same four fixed repairs. This loop makes each corridor's masked predictions and score rows once.

In [6]:
feb_parts = []
regime_maps = {}
base_targets = None
for panel in PANELS:
    frame, regime_map, target_map = evaluate_panel(panel, PROFILE_7, PROFILE_28, FEB_DAYS[panel])
    feb_parts.append(frame)
    regime_maps[panel] = regime_map
    if panel == BASE_PANEL:
        base_targets = target_map
    assert len(frame) // 5 == sum(len(target_map[d]) for d in FEB_DAYS[panel])
    del frame, target_map
all_results = pd.concat(feb_parts, ignore_index=True)
del feb_parts
target_method = 'Historical mean'
assert all_results.groupby(['panel','method']).size().unstack().nunique(axis=1).eq(1).all()
if VERIFY:
    assert len(all_results) // 5 == 1359715
print('Five February corridors | held-out days:', {p: len(FEB_DAYS[p]) for p in PANELS},
      '| paired target cells:', len(all_results)//5)

panel_regime = all_results.groupby(['panel','method','regime']).apply(
    component_score, include_groups=False).rename('score').reset_index()
panel_scores = panel_regime.pivot(index=['panel','method'], columns='regime', values='score')
panel_scores['macro'] = panel_scores[['R1','R2','R3']].mean(axis=1)
panel_counts = all_results[all_results.method.eq(target_method)].groupby(['panel','regime']).agg(
    days=('day','nunique'), cells=('day','size'))
expected_macro = {'D12_I405_N': {'Historical mean': 0.6425831902709835, 'Historical mean (28 days)': 0.8338954641422575, 'Spatial': 0.7750230332193918, 'Temporal': 0.9222214411818704, 'Temporal/spatial fallback': 0.912012688773579}, 'D12_I5_N': {'Historical mean': 0.3984577202117869, 'Historical mean (28 days)': 0.55434519355101, 'Spatial': 0.8434767565967375, 'Temporal': 0.9085525980511661, 'Temporal/spatial fallback': 0.9055389659387427}, 'D7_I405_N': {'Historical mean': 0.28413025787529644, 'Historical mean (28 days)': 0.5265567317317552, 'Spatial': 0.8403523875308471, 'Temporal': 0.8817628257454317, 'Temporal/spatial fallback': 0.881731304115834}, 'D12_I5_S': {'Historical mean': 0.4959169030626344, 'Historical mean (28 days)': 0.6332722523902637, 'Spatial': 0.7479113124186337, 'Temporal': 0.9050385239686757, 'Temporal/spatial fallback': 0.8919830906116785}, 'D7_I10_E': {'Historical mean': 0.5505121211872201, 'Historical mean (28 days)': 0.6715297674765006, 'Spatial': 0.8267615563508778, 'Temporal': 0.9111616958137386, 'Temporal/spatial fallback': 0.9038882453669421}}
if VERIFY:
    for panel in PANELS:
        for method, expected in expected_macro[panel].items():
            assert abs(panel_scores.loc[(panel, method), 'macro'] - expected) < 1e-12
            chart_key = 'Historical mean (7 days)' if method == 'Historical mean' else method
            assert abs(opening_scores[panel][chart_key] - expected) < 1e-6
display(panel_scores.round(4))
display(panel_counts.reset_index())

For paired differences, we resample complete held-out days within each regime and use the same resampled days for both methods. A 95% interval above zero is **better**, below zero **worse**, and crossing zero **no clear difference** for the first method. These describe this date slice; labels based on only 2–3 days in a regime are suggestive.

In [7]:
daily_sums = all_results.groupby(['panel','regime','day','method']).agg(
    speed_sq=('se_speed','sum'), flow_sq=('se_flow_lane','sum'), n=('day','size'))

def paired_day_interval(panel, first, second, repeats=1000, seed=7):
    rng = np.random.default_rng(seed)
    boot = np.zeros(repeats)
    for regime in ['R1','R2','R3']:
        group = daily_sums.xs((panel, regime), level=('panel','regime'))
        days = sorted(group.index.get_level_values('day').unique())
        picks = rng.integers(0, len(days), size=(repeats, len(days)))
        for sign, method in [(1, first), (-1, second)]:
            vals = group.xs(method, level='method').reindex(days)
            assert vals.notna().all().all()
            n = vals.n.to_numpy()[picks].sum(axis=1)
            sr = np.sqrt(vals.speed_sq.to_numpy()[picks].sum(axis=1)/n)
            fr = np.sqrt(vals.flow_sq.to_numpy()[picks].sum(axis=1)/n)
            score = .54*np.maximum(0,1-sr/25) + .46*np.maximum(0,1-fr/600)
            boot += sign*score/3
    return np.quantile(boot,[.025,.975])

comparisons = []
for panel in PANELS:
    for i,(first,second) in enumerate([
        ('Temporal','Historical mean'), ('Spatial','Historical mean'),
        ('Temporal/spatial fallback','Historical mean'),
        ('Temporal','Temporal/spatial fallback')]):
        point = panel_scores.loc[(panel,first),'macro'] - panel_scores.loc[(panel,second),'macro']
        seed = SEED if (panel in PANELS[3:] and i == 3) else SEED+i
        low, high = paired_day_interval(panel, first, second, seed=seed)
        day_counts = {r: int(panel_counts.loc[(panel,r),'days']) for r in ['R1','R2','R3']}
        comparisons.append({'corridor': panel, 'comparison': first+' vs '+second,
                            'difference': point, '95% low': low, '95% high': high,
                            'label': 'better' if low > 0 else ('worse' if high < 0 else 'no clear difference'),
                            'R1/R2/R3 days': '/'.join(str(day_counts[r]) for r in ['R1','R2','R3']),
                            'evidence': 'suggestive' if min(day_counts.values()) <= 3 else 'more days'})
comparison_table = pd.DataFrame(comparisons)
main_comparison = comparison_table[comparison_table.comparison.eq('Temporal vs Temporal/spatial fallback')]
if VERIFY:
    assert main_comparison.set_index('corridor').loc['D7_I405_N','label'] == 'no clear difference'
    assert main_comparison.set_index('corridor').label.eq('better').sum() == 4
display(comparison_table.round(4))

## Baseline strength: seven versus 28 earlier days

The historical profile uses the same weekday, five-minute slot, link, quality rule, target cells and scoring formula at both history lengths. Only its earlier window changes: seven days (2031-02-02 through 2031-02-08) versus 28 days (2031-01-12 through 2031-02-08). The 28-day baseline is stronger but still shorter than the organizers' full-data baseline. A separate D12_I405_N January backtest uses January 13–19 or December 23–January 19 for profiles, then January 20–31 as held-out days.

In [8]:
history_comparison = pd.DataFrame([
    {'corridor': p, 'Historical 7 days': panel_scores.loc[(p,'Historical mean'),'macro'],
     'Historical 28 days': panel_scores.loc[(p,'Historical mean (28 days)'),'macro'],
     'Temporal': panel_scores.loc[(p,'Temporal'),'macro'],
     'Temporal minus 28-day mean': panel_scores.loc[(p,'Temporal'),'macro'] -
                                  panel_scores.loc[(p,'Historical mean (28 days)'),'macro']}
    for p in PANELS])
if VERIFY:
    assert (history_comparison['Historical 28 days'] > history_comparison['Historical 7 days']).all()
display(history_comparison.round(4))

jan_frame, jan_regimes, _ = evaluate_panel(BASE_PANEL, JAN_7, JAN_28, JAN_HELD)
jan_regime = jan_frame.groupby(['method','regime']).apply(
    component_score, include_groups=False).rename('score').reset_index()
jan_scores = jan_regime.pivot(index='method', columns='regime', values='score')
jan_scores['macro'] = jan_scores[['R1','R2','R3']].mean(axis=1)
jan_expected = {'Historical mean': .6299876329050679,
                'Historical mean (28 days)': .7689209478148049,
                'Spatial': .7783821819359559,
                'Temporal': .9256607022548989,
                'Temporal/spatial fallback': .9128330759524316}
if VERIFY:
    for method, expected in jan_expected.items():
        assert abs(jan_scores.loc[method, 'macro'] - expected) < 1e-12
    assert len(jan_frame)//5 == 74431
    assert [list(jan_regimes.values()).count(r) for r in ['R1','R2','R3']] == [4,5,3]
print('Separate January backtest | 12 days | 74,431 paired target cells')
display(jan_scores.round(4))
del jan_frame

## Gap length and day-edge cells

A run length counts consecutive null speed readings on the same link **within the masked day**. Short means 1–2 slots, medium 3–6, and long 7 or more. Only official template targets enter these tables. A visible null run can include unscored missingness or a Task 2 buffer, so its length does not prove a detector outage. The fallback equals Temporal on short runs and Spatial on long runs, per channel. A Temporal edge flag marks a target without an observed same-day reading on both sides for at least one channel; that channel uses the earlier profile. No neighboring held-out day is used.

In [9]:
labels = ['short (1–2)', 'medium (3–6)', 'long (7+)']
all_results['gap_bucket'] = pd.cut(all_results.gap_steps, bins=[0,2,6,np.inf], labels=labels)
assert all_results.gap_bucket.notna().all()
gap_scores = all_results.groupby(['panel','gap_bucket','method'], observed=True).apply(
    component_score, include_groups=False).rename('score').reset_index()
gap_table = gap_scores.pivot(index=['panel','gap_bucket'], columns='method', values='score')
gap_counts = all_results[all_results.method.eq('Historical mean')].groupby(
    ['panel','gap_bucket'], observed=True).size().rename('cells')
gap_table = gap_table.join(gap_counts)
gap_table['Temporal minus fallback'] = gap_table['Temporal'] - gap_table['Temporal/spatial fallback']
short_scores = gap_table.xs(labels[0], level='gap_bucket')['Temporal']
long_scores = gap_table.xs(labels[-1], level='gap_bucket')['Temporal']
gap_table['short minus long'] = np.nan
for panel in PANELS:
    gap_table.loc[(panel, labels[-1]), 'short minus long'] = short_scores[panel] - long_scores[panel]
if VERIFY:
    assert (short_scores - long_scores).gt(0).all()
display(gap_table.round(4))

target_gaps = all_results[all_results.method.eq('Historical mean')]
edge_table = target_gaps.groupby(['panel','gap_bucket'], observed=True).agg(
    target_cells=('edge_any','size'), edge_cells=('edge_any','sum'),
    speed_edge_cells=('speed_kmh_edge','sum'), flow_edge_cells=('flow_vph_edge','sum'))
edge_table['edge_share_pct'] = 100*edge_table.edge_cells/edge_table.target_cells
display(edge_table.reset_index().round(2))

for bucket in labels[:2]:
    rows = gap_table.xs(bucket, level='gap_bucket')
    assert np.allclose(rows['Temporal'], rows['Temporal/spatial fallback'], atol=1e-12)
long_rows = all_results[all_results.gap_bucket.eq(labels[-1])]
long_by_panel = long_rows.groupby(['panel','method']).apply(
    component_score, include_groups=False).unstack()
assert np.allclose(long_by_panel['Spatial'], long_by_panel['Temporal/spatial fallback'], atol=1e-12)
long_counts = target_gaps[target_gaps.gap_bucket.eq(labels[-1])].groupby('panel').size()
mixed_channels = (target_gaps[target_gaps.gap_bucket.eq(labels[-1])]
                  .assign(mixed=lambda x: x.flow_gap_steps <= 6)
                  .groupby('panel').mixed.sum().reindex(PANELS, fill_value=0))
if VERIFY:
    assert mixed_channels.loc[['D12_I5_S','D7_I10_E']].eq(0).all()
    assert mixed_channels.eq(0).all()
long_comparison = pd.DataFrame({
    'target cells': long_counts,
    'Temporal': long_by_panel['Temporal'],
    'Spatial branch': long_by_panel['Temporal/spatial fallback'],
    'Temporal minus Spatial': long_by_panel['Temporal'] - long_by_panel['Temporal/spatial fallback'],
    'long speed / short flow': mixed_channels})
if VERIFY:
    assert int(long_comparison['Temporal minus Spatial'].gt(0).sum()) == 4
    assert int(long_comparison.loc['D12_I5_S','target cells']) == 11149
    assert int(long_comparison.loc['D7_I10_E','target cells']) == 15932
    assert abs(long_comparison.loc['D12_I5_S','Temporal'] - .8669981803084956) < 1e-12
    assert abs(long_comparison.loc['D7_I10_E','Temporal'] - .88023491291452) < 1e-12
display(long_comparison.round(4))
print('Temporal leads the long-gap Spatial branch in 4 of 5 corridors; the two added corridors have zero mixed-channel long-gap targets.')

first_three_long = long_rows[long_rows.panel.isin(PANELS[:3])]
pooled_three = first_three_long.groupby('method').apply(component_score, include_groups=False)
if VERIFY:
    assert len(first_three_long)//5 == 60352
    assert abs(pooled_three['Temporal'] - .8584630896801965) < 1e-12
    assert abs(pooled_three['Temporal/spatial fallback'] - .8425861677575276) < 1e-12

sensitivity = []
for panel in PANELS + ['First three pooled']:
    section = first_three_long if panel == 'First three pooled' else long_rows[long_rows.panel.eq(panel)]
    eligible = section[~section.edge_any]
    scores = eligible.groupby('method').apply(component_score, include_groups=False)
    sensitivity.append({'corridor': panel, 'target cells': len(eligible)//5,
                        'Temporal': scores['Temporal'],
                        'Spatial branch': scores['Temporal/spatial fallback']})
long_sensitivity = pd.DataFrame(sensitivity).set_index('corridor')
if VERIFY:
    assert abs(long_sensitivity.loc['D7_I405_N','Temporal'] - .8590551213771441) < 1e-12
    assert abs(long_sensitivity.loc['D7_I405_N','Spatial branch'] - .854448791233881) < 1e-12
display(long_sensitivity.round(4))

## One day-edge example

The first D7_I405_N long speed gap at a same-day edge, ordered by timestamp and link, was selected without looking at prediction error. The chart shows visible same-day speed, the repairs, and train truth for explanation. Temporal has no observation bracketing that target within the day and uses the earlier profile. The fallback uses Spatial on this eight-step gap. One trace explains a mechanism; it is not a representative score estimate.

In [10]:
example_pool = target_gaps[(target_gaps.panel.eq('D7_I405_N')) &
                           (target_gaps.gap_steps >= 7) & target_gaps.speed_kmh_edge]
example_key = example_pool.sort_values(['day','timestamp','link_id']).iloc[0]
example_day = example_key.day
example_regime = regime_maps['D7_I405_N'][example_day]
example_masked = read_day('D7_I405_N', example_day, True, example_regime)
example_profile, example_link_mean = profile_for('D7_I405_N', PROFILE_7)
example_order = (example_masked[['link_id','milepost']].drop_duplicates()
                 .sort_values('milepost').link_id.tolist())
example_repairs = repair_masked(example_masked, example_order, example_profile, example_link_mean)
at_target = example_masked.timestamp.eq(example_key.timestamp) & example_masked.link_id.eq(example_key.link_id)
assert at_target.sum() == 1
row_index = np.flatnonzero(at_target.to_numpy())[0]
example_truth = pd.read_parquet(day_path('D7_I405_N', example_day, False), columns=KEY+CHANNELS)
truth_row = example_truth[(example_truth.timestamp.eq(example_key.timestamp)) &
                          (example_truth.link_id.eq(example_key.link_id))].iloc[0]
example_values = pd.DataFrame([
    {'value': method, 'speed_kmh': float(pred.iloc[row_index].speed_kmh),
     'flow_vph': float(pred.iloc[row_index].flow_vph)}
    for method, pred in example_repairs.items()
] + [{'value': 'Train truth (scoring only)',
      'speed_kmh': float(truth_row.speed_kmh), 'flow_vph': float(truth_row.flow_vph)}])
if VERIFY:
    assert abs(example_values.loc[example_values.value.eq('Temporal'),'speed_kmh'].iloc[0] -
               example_values.loc[example_values.value.eq('Historical mean'),'speed_kmh'].iloc[0]) < 1e-12
display(example_values.round(2))

link_day = example_masked[example_masked.link_id.eq(example_key.link_id)].sort_values('timestamp')
ix = np.flatnonzero(link_day.timestamp.eq(example_key.timestamp).to_numpy())[0]
window = link_day.iloc[max(0, ix-8):min(len(link_day), ix+17)]
times = pd.to_datetime(window.timestamp, utc=True)
target_time = pd.to_datetime(example_key.timestamp, utc=True)
fig, ax = plt.subplots(figsize=(8.5, 3.7))
ax.plot(times, window.speed_kmh, 'o-', color='#596b72', markersize=3,
        label='Visible same-day speed')
palette = {'Temporal':'#147d74', 'Spatial':'#c36a3e',
           'Historical mean':'#526aa6', 'Train truth (scoring only)':'#222222'}
for _, value in example_values.iterrows():
    if value['value'] == 'Temporal/spatial fallback':
        continue
    ax.scatter(target_time, value.speed_kmh, s=64, color=palette[value['value']],
               marker='D' if value['value'] == 'Train truth (scoring only)' else 'o',
               label=value['value'], zorder=5)
ax.set_title('At this day edge, Temporal uses the earlier profile')
ax.set_ylabel('Speed (km/h)')
ax.set_xlabel('Same-day five-minute readings; blank points are masked')
ax.legend(loc='upper left', bbox_to_anchor=(1.01,1), frameon=False, fontsize=8)
ax.spines[['top','right']].set_visible(False)
fig.autofmt_xdate()
plt.tight_layout(); plt.show()

## Reuse recipe

After Setup and Repairs, provide one masked day and an earlier eligible profile. Order links by the released milepost, call `repair_masked`, then join the official template keys to inspect target predictions. For another corridor, change the panel and chronological date windows in Setup. This preview reads no truth and writes no submission file.

In [11]:
day_to_repair = FEB_DAYS[BASE_PANEL][0]
masked_day = read_day(BASE_PANEL, day_to_repair, True, regime_maps[BASE_PANEL][day_to_repair])
link_order = (masked_day[['link_id','milepost']].drop_duplicates()
              .sort_values('milepost').link_id.tolist())
reusable_profile, reusable_mean = profile_for(BASE_PANEL, PROFILE_7)
temporal_day = repair_masked(masked_day, link_order, reusable_profile, reusable_mean)['Temporal']
day_keys = base_targets[day_to_repair][KEY]
target_preview = masked_day[KEY].join(temporal_day).merge(day_keys, on=KEY, validate='one_to_one')
assert len(target_preview) == len(day_keys)
display(target_preview.head(5).round(2))

## Published-scorer check

This table compares this clean notebook run with **recorded values from a separate local run** of the published [Task 1 scorer](https://github.com/jacky850/trafficflowbench-public/blob/main/src/task1/score_task1.py), SHA-256 `bcd42e88fbc42cd499e93e3b50ddf367009b26b216df5ab3f710ee1d1d863163`. That local run scored all 1,434,146 selected Temporal target cells with zero missing predictions. Only its file enumerators were restricted to the declared dates; target joins, lane lookup, score arithmetic, and aggregation remained unchanged. The table is a recorded-value comparison; this Kaggle run does not execute the published scorer.

To reproduce it, obtain that scorer source and the same rebuilt release; generate Temporal predictions for the five February corridors and D12_I405_N January dates, restrict `release_files` and `masked_files` to those exact daily partitions, and call `score_panel` for each panel/window. The local `official_subset_check.py` in this project implements those file selections and writes the recorded R1/R2/R3 and macro results. A full ten-direction competition score is outside this subset.

In [12]:
official_temporal = {'D12_I405_N': {'R1': 0.9341020310351212, 'R2': 0.914599060706162, 'R3': 0.9179632318043283, 'macro_mean': 0.9222214411818704}, 'D12_I5_N': {'R1': 0.9116886087471194, 'R2': 0.9158040230487867, 'R3': 0.8981651623575918, 'macro_mean': 0.9085525980511661}, 'D7_I405_N': {'R1': 0.8904795637664189, 'R2': 0.869440593323375, 'R3': 0.8853683201465015, 'macro_mean': 0.8817628257454317}, 'D12_I5_S': {'R1': 0.9062728138019917, 'R2': 0.9091212698613665, 'R3': 0.8997214882426687, 'macro_mean': 0.9050385239686757}, 'D7_I10_E': {'R1': 0.9133554773025099, 'R2': 0.9216227681844247, 'R3': 0.8985068419542814, 'macro_mean': 0.9111616958137386}, 'January D12_I405_N': {'R1': 0.9249224633285511, 'R2': 0.9307742182321367, 'R3': 0.9212854252040089, 'macro_mean': 0.9256607022548989}}
scorer_rows = []
for panel in PANELS:
    for regime in ['R1','R2','R3','macro_mean']:
        ours = panel_scores.loc[(panel,'Temporal'), 'macro' if regime == 'macro_mean' else regime]
        scorer_rows.append({'corridor':panel, 'regime':regime,
                            'absolute difference':abs(ours-official_temporal[panel][regime])})
for regime in ['R1','R2','R3','macro_mean']:
    ours = jan_scores.loc['Temporal', 'macro' if regime == 'macro_mean' else regime]
    scorer_rows.append({'corridor':'January D12_I405_N', 'regime':regime,
                        'absolute difference':abs(ours-official_temporal['January D12_I405_N'][regime])})
scorer_check = pd.DataFrame(scorer_rows)
if VERIFY:
    assert scorer_check['absolute difference'].max() < 1e-12
print('Published scorer target cells: 1,434,146 | missing predictions: 0')
display(scorer_check.groupby('corridor', sort=False)['absolute difference'].max().to_frame().round(12))

## Limits and takeaways

- Temporal **leads four corridors and ties with fallback on D7_I405_N** by paired day-block uncertainty. Labels for D12_I405_N and D12_I5_N involve only 2–3 days in a regime and are suggestive.
- A 28-day historical profile improves every seven-day baseline on the same targets, narrowing Temporal's apparent advantage. The organizers' full-data baseline remains untested.
- For long gaps, the fallback switches to Spatial. Temporal leads that branch on **four of five** corridors; D7_I405_N reverses 0.8530 versus 0.8551. Excluding its flagged day-edge cells changes the ordering to 0.8591 versus 0.8544. The two added corridors have no long speed-gap target with a short flow gap.
- The separate January backtest scores Temporal 0.9257 versus 0.7689 for the 28-day historical mean. Recorded values from the published scorer match every selected Temporal score within floating-point precision.

Repairs use only the same day's masked file plus earlier profile observations. This is synthetic train-date evidence on five of ten directional corridors and one earlier January window. Other seasons, the full organizer baseline, Tasks 2–4, and an overall competition score remain untested.